# Pre-process MovieLens

Same pipeline as `process.ipynb` (dedup → rating filter → k-core → map ids → leave-one-out split → product table → simulator jsonl), adapted for the MovieLens schema: plain `.csv` instead of `.jsonl.gz`, join on `movieId`, title/year regex split, pipe-delimited `genres` instead of a JSON `categories` list.

# 0. Import & logging

In [1]:
import os
import re
import json
import pickle
import logging
from local_package.config.data import MOVIELENS_RAW_DIR, MOVIELENS_PROCESSED_DIR
from local_package.config.log import setup_logging, MOVIELENS_PROCESS_LOG_DIR
import pandas as pd

In [2]:
logger = setup_logging(name="process", level=logging.INFO, to_file=True, log_dir=MOVIELENS_PROCESS_LOG_DIR)

# 1. Configuration

In [3]:
# Seed
SEED = 2024

In [4]:
# Path
DATA_DIR = MOVIELENS_RAW_DIR  # swap for any MovieLens release with the same movies.csv/ratings.csv schema

MOVIES_FILE = DATA_DIR / "movies.dat"
RATINGS_FILE = DATA_DIR / "ratings.dat"

MOVIE_COLUMN_NAME = ["MovieID", "Title", "Genres"]
RATINGS_COLUMN_NAME = ["UserID", "MovieID", "Rating", "Timestamp"]

OUTPUT_DIR = MOVIELENS_PROCESSED_DIR / "chatbot"  # where train/valid/test/products/simulator files go
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Rating threshold for filtering ratings
RATING_THRESHOLD = 3.0  # rows below this rating are dropped

# Min interactions per user/item for k-core filtering
USER_K = 5  # k-core: min interactions per user
ITEM_K = 5  # k-core: min interactions per item

# Maximum lengths for simulator strings
MAX_HISTORY_LEN = 10  # max past items shown in a simulator history string
MAX_TITLE_LEN = 50  # max chars of a movie title used in simulator strings

# Simulator sampling
SIMULATOR_SAMPLE_N = 900  # test users sampled for the simulator jsonl export

# 2. Load raw data

In [6]:
def load_movies_and_ratings(movies_file, ratings_file):
    logger.info("Loading movies from %s", movies_file)
    movies_df = pd.read_csv(
        movies_file,
        sep="::",
        header=None,
        names=MOVIE_COLUMN_NAME,
        engine="python",
    )

    logger.info("Loading ratings from %s", ratings_file)
    ratings_df = pd.read_csv(
        ratings_file,
        sep="::",
        header=None,
        names=RATINGS_COLUMN_NAME,
        engine="python",
    )

    logger.info("Shape of movies: %s", movies_df.shape)
    logger.info("Shape of ratings: %s", ratings_df.shape)
    return movies_df, ratings_df

In [7]:
movies_df, ratings_df = load_movies_and_ratings(MOVIES_FILE, RATINGS_FILE)

2026-09-07 04:27:27 [INFO] process: Loading movies from C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\raw\ml\movies.dat
2026-09-07 04:27:27 [INFO] process: Loading ratings from C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\raw\ml\ratings.dat
2026-09-07 04:28:20 [INFO] process: Shape of movies: (10681, 3)
2026-09-07 04:28:20 [INFO] process: Shape of ratings: (10000054, 4)


# 3. Clean metadata

In [8]:
def extract_title_and_year(titles, pattern=r"^(.+)\s\((\d{4})\)$"):
    # MovieLens bakes the year into the title, e.g. "Toy Story (1995)"; a handful of titles
    # (mostly foreign-language or mis-scraped entries) have no trailing year at all
    extracted = titles.str.extract(pattern)
    title = extracted[0].fillna(titles).str.strip()
    release_date = pd.to_datetime(extracted[1], format="%Y", errors="coerce")
    return title, release_date

In [9]:
def join_genres(x):
    # genres come pipe-separated, e.g. "Adventure|Animation|Comedy"; MovieLens uses the literal
    # string "(no genres listed)" for movies with none, analogous to Amazon's empty description list
    if not isinstance(x, str) or x == "(no genres listed)":
        return "No description"
    return ", ".join(x.split("|"))

In [10]:
movies_df = movies_df[~movies_df["Title"].isna()].reset_index(drop=True)
logger.info("Movies after dropping missing titles: %s", movies_df.shape)

2026-09-07 04:28:20 [INFO] process: Movies after dropping missing titles: (10681, 3)


In [11]:
movies_df["Title"], movies_df["release_date"] = extract_title_and_year(movies_df["Title"])
movies_df["category"] = movies_df["Genres"].apply(
    lambda x: x.split("|")[0] if isinstance(x, str) and x != "(no genres listed)" else "Unknown"
)
movies_df["description"] = movies_df["Genres"].apply(join_genres)

# 4. Filter out ratings

In [12]:
def get_valid_ids(df, col_name, k):
    frequency = df.groupby([col_name])[[col_name]].count()
    return frequency[frequency[col_name] >= k].index

In [13]:
def keep_first_filter(df, user_col="UserID", item_col="MovieID", time_col="Timestamp"):
    logger.info("Keeping first interaction per duplicated rating, begin: %s", df.shape)
    df = df.sort_values(by=[user_col, time_col]).reset_index(drop=True)
    df = df.drop_duplicates(subset=[user_col, item_col], keep="first").reset_index(drop=True)
    logger.info("After keep-first filter: %s", df.shape)
    return df

In [14]:
def low_rating_filter(df, rating_thres=RATING_THRESHOLD, rating_col="Rating"):
    logger.info("Filtering ratings below %.1f, begin: %s", rating_thres, df.shape)
    df = df[df[rating_col] >= rating_thres].reset_index(drop=True)
    logger.info("After rating filter: %s", df.shape)
    return df

In [15]:
def k_core_filter(df, user_k=USER_K, item_k=ITEM_K, user_col="UserID", item_col="MovieID", max_iter=20):
    logger.info("k-core filtering (user_k=%d, item_k=%d), begin: %s", user_k, item_k, df.shape)
    num_users_prev, num_items_prev = len(df[user_col].unique()), len(df[item_col].unique())
    delta, it = True, 0
    while delta and it < max_iter:
        valid_users = get_valid_ids(df, user_col, user_k)
        df = df[df[user_col].isin(valid_users)]
        valid_items = get_valid_ids(df, item_col, item_k)
        df = df[df[item_col].isin(valid_items)]
        num_users, num_items = len(valid_users), len(valid_items)
        delta = (num_users != num_users_prev) or (num_items != num_items_prev)
        logger.info("Iter %d: users %d/%d, items %d/%d", it, num_users, num_users_prev, num_items, num_items_prev)
        num_users_prev, num_items_prev = num_users, num_items
        it += 1
    logger.info("After k-core filter: %s", df.shape)
    return df

In [16]:
ratings_df = ratings_df[ratings_df["MovieID"].isin(movies_df["MovieID"])].reset_index(drop=True)
data_df = keep_first_filter(ratings_df)
data_df = low_rating_filter(data_df)
data_df = k_core_filter(data_df).reset_index(drop=True)

2026-09-07 04:28:20 [INFO] process: Keeping first interaction per duplicated rating, begin: (10000054, 4)
2026-09-07 04:28:28 [INFO] process: After keep-first filter: (10000054, 4)
2026-09-07 04:28:28 [INFO] process: Filtering ratings below 3.0, begin: (10000054, 4)
2026-09-07 04:28:28 [INFO] process: After rating filter: (8242124, 4)
2026-09-07 04:28:28 [INFO] process: k-core filtering (user_k=5, item_k=5), begin: (8242124, 4)
2026-09-07 04:28:29 [INFO] process: Iter 0: users 69814/69863, items 9888/10598
2026-09-07 04:28:30 [INFO] process: Iter 1: users 69814/69814, items 9888/9888
2026-09-07 04:28:30 [INFO] process: After k-core filter: (8240192, 4)


# 5. Mapping users and items' IDs

In [17]:
def map_id(df, user_colname="UserID", item_colname="MovieID"):
    logger.info("Mapping user and item ids to contiguous integers")
    users, items = df[user_colname].unique(), df[item_colname].unique()
    user_map = {u: k + 1 for k, u in enumerate(users)}
    item_map = {i: k + 1 for k, i in enumerate(items)}
    df[user_colname] = df[user_colname].apply(lambda x: user_map[x])
    df[item_colname] = df[item_colname].apply(lambda x: item_map[x])
    return df, user_map, item_map

In [18]:
data_df, user_map, item_map = map_id(data_df)

2026-09-07 04:28:30 [INFO] process: Mapping user and item ids to contiguous integers


In [19]:
# Save the id maps to a json file for later use
# (json keys must be strings; MovieLens ids are ints, so cast them going in)
with open(os.path.join(OUTPUT_DIR, "map.json"), "w") as f:
    json.dump(
        {"item": {str(k): v for k, v in item_map.items()}, "user": {str(k): v for k, v in user_map.items()}}, f
    )
logger.info("Saved id maps to %s", os.path.join(OUTPUT_DIR, "map.json"))

2026-09-07 04:28:39 [INFO] process: Saved id maps to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\ml\chatbot\map.json


# 6. Leave-one-out split

In [20]:
def split_leave_one_out_seq(data, col_name, time_colname, col_names_2_return):
    df_sorted = data.sort_values(by=[col_name, time_colname]).reset_index(drop=True)
    df_test = df_sorted.groupby(by=col_name, as_index=False).nth(-1)
    df_train = df_sorted.iloc[df_sorted.index.difference(df_test.index)]
    return (
        df_train.reset_index(drop=True)[col_names_2_return],
        df_test.reset_index(drop=True)[col_names_2_return],
    )

In [21]:
df_train_0, df_test = split_leave_one_out_seq(data_df, "UserID", "Timestamp", ["UserID", "MovieID", "Timestamp"])
df_train, df_valid = split_leave_one_out_seq(df_train_0, "UserID", "Timestamp", ["UserID", "MovieID"])

In [22]:
df_train.to_csv(os.path.join(OUTPUT_DIR, "train.tsv"), index=None)
df_valid.to_csv(os.path.join(OUTPUT_DIR, "valid.tsv"), index=None)
df_test.to_csv(os.path.join(OUTPUT_DIR, "test.tsv"), index=None)
df_train_0.to_csv(os.path.join(OUTPUT_DIR, "user_history.tsv"), index=None)

logger.info(
    "Saved splits to %s (train=%d, valid=%d, test=%d, full_history=%d)",
    OUTPUT_DIR, len(df_train), len(df_valid), len(df_test), len(df_train_0),
)

2026-09-07 04:29:04 [INFO] process: Saved splits to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\ml\chatbot (train=8100564, valid=69814, test=69814, full_history=8170378)


# 7. Product table

Named `products.*` (not `movies.*`) to match the Amazon output layout, so downstream chatbot/recsys code can read either domain's processed folder the same way.

In [23]:
saved_meta_df = movies_df[movies_df["MovieID"].isin(item_map.keys())]
saved_meta_df = saved_meta_df.drop_duplicates(subset=["MovieID"], keep="first").reset_index(drop=True)
saved_meta_df["MovieID"] = saved_meta_df["MovieID"].apply(lambda x: item_map[x])

In [24]:
user_history = df_train_0.groupby("UserID").agg(list)
item_count = user_history["MovieID"].explode().value_counts()
saved_meta_df.rename(columns={"MovieID": "id"}, inplace=True)
saved_meta_df["visited_num"] = saved_meta_df["id"].apply(lambda x: item_count.loc[x] if x in item_count else 0)

In [25]:
saved_meta_df.to_feather(os.path.join(OUTPUT_DIR, "products.ftr"))
saved_meta_df.to_csv(os.path.join(OUTPUT_DIR, "products.csv"), index=None, sep="|")
logger.info("Saved product table (%d items) to %s", len(saved_meta_df), OUTPUT_DIR)

2026-09-07 04:29:10 [INFO] process: Saved product table (9888 items) to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\ml\chatbot


# 8. Simulator jsonl sample

In [26]:
def write_jsonl(obj, fpath):
    try:
        with open(fpath, "w") as outfile:
            for entry in obj:
                json.dump(entry, outfile)
                outfile.write("\n")
        logger.info("Saved %d records to %s", len(obj), fpath)
    except Exception as e:
        fallback = f"{fpath}.pkl"
        logger.exception("Failed to write jsonl (%s), falling back to pickle at %s", e, fallback)
        with open(fallback, "wb") as tempfile:
            pickle.dump(obj, tempfile)

In [27]:
saved_meta_df_indexed = saved_meta_df.set_index("id")
id2title = {id_: saved_meta_df_indexed.loc[id_].Title[:MAX_TITLE_LEN] for id_ in saved_meta_df_indexed.index}

In [28]:
n_sample = min(SIMULATOR_SAMPLE_N, len(df_test))
test_data = df_test.sample(n_sample, random_state=SEED)
test_data["history"] = test_data["UserID"].apply(
    lambda x: "; ".join([id2title[i] for i in user_history.loc[x]["MovieID"][-MAX_HISTORY_LEN:]])
)
test_data["target"] = test_data["MovieID"].apply(lambda x: saved_meta_df_indexed.loc[x].Title)
test_data.reset_index(drop=True, inplace=True)

In [29]:
simulator_path = os.path.join(OUTPUT_DIR, f"simulator_test_data_{n_sample}.jsonl")
write_jsonl(test_data[["history", "target"]].to_dict("records"), simulator_path)
logger.info("Pipeline complete.")

2026-09-07 04:29:12 [INFO] process: Saved 900 records to C:\Users\i_am_fuch\Desktop\agentic-rag-for-rcm-sys\data\processed\ml\chatbot\simulator_test_data_900.jsonl
2026-09-07 04:29:12 [INFO] process: Pipeline complete.
